# Isolated-fish frames

For each fish, find every frame in which it is **at least `ISO_FRAC` (here 1/4) of the tank side length away from all other fish**.

* **Distance** = edge-to-edge: the shortest distance between any pixel on the outline of one fish's mask and any pixel on another fish's mask. (Centroid distance would understate how close two long fish can be.)
* **Tank side**: the inner tank walls span ≈ 840 px of the 912 px video frame (measured on the review videos). Change `TANK_SIDE_PX` if your tank fills the frame differently.
* A frame counts only when **all four fish are tracked**. If a fish is missing, you can't tell whether it's close by, so that frame is left out for every fish.
* Remnant masks (label 4) are ignored.
* Masks come from either a ZEUS `*_tracking_final.pkl` (labels = fish ranked by size) or a packed full-frame `.mgz` (labels = tracker object ids 0-3). This notebook is set up for the recording used in `eod_two_layer_pipeline_compare_electrode.ipynb`, which has EOD recordings (last section).

In [ ]:
import gzip
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.ndimage import binary_erosion
from scipy.spatial import cKDTree

REC = "video2024-02-20T21_16_26"       # recording used in eod_two_layer_pipeline_compare_electrode.ipynb
MASKS = f"data/{REC}__223e__59997__masks_packed.mgz"   # or e.g. "data/elephantnose_18_16_24_tracking_final.pkl"
TANK_SIDE_PX = 840                     # inner tank side in video pixels
ISO_FRAC = 1 / 4                       # isolation distance as a fraction of the tank side
MIN_DIST = TANK_SIDE_PX * ISO_FRAC     # isolation threshold (px)
TAG = f"{REC}_iso1-{round(1 / ISO_FRAC)}"   # prefix for saved results

FISH = [0, 1, 2, 3]
COLORS = {1: "#e87ba4", 2: "#008300", 3: "#eda100", 0: "#2a78d6"}

out = Path("results/isolated_frames")
out.mkdir(parents=True, exist_ok=True)


def load_mgz(path, fps=50.0):
    """Packed full-frame .mgz masks -> the *_tracking_final.pkl layout (cropped, bit-packed masks)."""
    frames = {}
    with gzip.open(path, "rb") as f:
        pickle.load(f)                                    # header
        while True:
            try:
                k, objs = pickle.load(f)
            except EOFError:
                break
            fr = frames[k] = {"labels": [], "boxes": [], "masks": [], "areas": []}
            for oid, (shape, packed) in sorted(objs.items()):
                m = np.unpackbits(packed, count=shape[0] * shape[1]).reshape(shape)
                rows, cols = np.flatnonzero(m.any(1)), np.flatnonzero(m.any(0))
                if rows.size == 0:
                    continue
                y0, y1, x0, x1 = int(rows[0]), int(rows[-1]) + 1, int(cols[0]), int(cols[-1]) + 1
                fr["labels"].append(oid)
                fr["boxes"].append((y0, y1, x0, x1))
                fr["masks"].append(np.packbits(m[y0:y1, x0:x1]))
                fr["areas"].append(int(m[y0:y1, x0:x1].sum()))
    return {"fps": fps, "shape": tuple(shape), "frames": frames}


def load_tracking(path):
    if path.endswith(".pkl"):
        with open(path, "rb") as f:
            return pickle.load(f)
    cache = out / f"{REC}_masks_cropped.pkl"             # decoding the .mgz takes a few minutes
    if not cache.exists():
        with open(cache, "wb") as f:
            pickle.dump(load_mgz(path), f)
    with open(cache, "rb") as f:
        return pickle.load(f)


d = load_tracking(MASKS)
fps, n_frames = d["fps"], len(d["frames"])
NAMES = {l: f"fish {l}" + (f" ({d['labels_key'][l]})" if "labels_key" in d else "") for l in FISH}
print(f"{REC}: {n_frames} frames at {fps:g} fps, threshold {MIN_DIST:.0f} px")

## Pairwise edge-to-edge distance per frame

In [ ]:
def outline(packed, box):
    """(N, 2) frame coordinates (y, x) of the mask's boundary pixels."""
    y0, y1, x0, x1 = box
    crop = np.unpackbits(packed)[: (y1 - y0) * (x1 - x0)].reshape(y1 - y0, x1 - x0).astype(bool)
    edge = crop & ~binary_erosion(crop)
    return np.argwhere(edge) + (y0, x0)


def box_gap(a, b):
    """Distance between two (y0, y1, x0, x1) boxes - a lower bound on the mask distance."""
    dy = max(a[0] - b[1], b[0] - a[1], 0)
    dx = max(a[2] - b[3], b[2] - a[3], 0)
    return np.hypot(dy, dx)


def frame_distances(fr):
    """4x4 matrix of edge-to-edge distances (FISH order), or None if a fish is missing."""
    idx = {lab: i for i, lab in enumerate(fr["labels"]) if lab != 4}
    if sorted(idx) != sorted(FISH):
        return None
    D = np.full((4, 4), np.inf)
    pts = {}
    for a in range(4):
        for b in range(a + 1, 4):
            ia, ib = idx[FISH[a]], idx[FISH[b]]
            gap = box_gap(fr["boxes"][ia], fr["boxes"][ib])
            if gap < MIN_DIST:   # boxes are close: measure the masks exactly
                for i in (ia, ib):
                    if i not in pts:
                        pts[i] = outline(fr["masks"][i], fr["boxes"][i])
                gap = cKDTree(pts[ia]).query(pts[ib])[0].min()
            D[a, b] = D[b, a] = gap
    return D


dist = np.full((n_frames, 4, 4), np.nan)
for k, fr in d["frames"].items():
    D = frame_distances(fr)
    if D is not None:
        dist[k] = D

all_tracked = ~np.isnan(dist[:, 0, 1])
print(f"frames with all four fish tracked: {all_tracked.sum()} / {n_frames}")

## Isolated frames per fish

`nearest[k, i]` is fish *i*'s distance to its closest neighbour in frame *k* (NaN when not all fish are tracked). When the two boxes are already at least `MIN_DIST` apart, the exact distance isn't computed: the value is the gap between the boxes, which is still at least `MIN_DIST`, so the yes/no decision is exact.

In [ ]:
nearest = dist.min(axis=2)        # diagonal is inf; NaN where not all fish are tracked
isolated = {lab: np.flatnonzero(nearest[:, i] >= MIN_DIST) for i, lab in enumerate(FISH)}

pd.DataFrame({
    "fish": [NAMES[l] for l in FISH],
    "isolated frames": [len(isolated[l]) for l in FISH],
    "% of tracked frames": [100 * len(isolated[l]) / all_tracked.sum() for l in FISH],
}).round(1)

Group consecutive frames into isolation episodes:

In [ ]:
def episodes(frames):
    """Runs of consecutive frame indices as (start, end) inclusive."""
    if len(frames) == 0:
        return pd.DataFrame(columns=["start", "end", "n_frames", "duration_s"])
    breaks = np.flatnonzero(np.diff(frames) > 1)
    starts = frames[np.r_[0, breaks + 1]]
    ends = frames[np.r_[breaks, len(frames) - 1]]
    n = ends - starts + 1
    return pd.DataFrame({"start": starts, "end": ends, "n_frames": n, "duration_s": n / fps})


eps = pd.concat({NAMES[l]: episodes(isolated[l]) for l in FISH}, names=["fish", "episode"])
eps.groupby("fish", sort=False).duration_s.describe().round(2)

## Overview

In [ ]:
t = np.arange(n_frames) / fps / 60
fig, axes = plt.subplots(len(FISH), 1, figsize=(12, 6), sharex=True, sharey=True)
for ax, (i, lab) in zip(axes, enumerate(FISH)):
    ax.plot(t, nearest[:, i], color=COLORS[lab], lw=0.4)
    ax.fill_between(t, 0, 1, where=nearest[:, i] >= MIN_DIST, color=COLORS[lab],
                    alpha=0.2, transform=ax.get_xaxis_transform(), lw=0)
    ax.axhline(MIN_DIST, color="k", ls="--", lw=0.8)
    ax.set_ylim(bottom=0)
    ax.set_ylabel(NAMES[lab], rotation=0, ha="right", va="center")
    ax.spines[["top", "right"]].set_visible(False)
axes[-1].set_xlabel("time (min)")
fig.suptitle(f"{REC} - distance to nearest fish (px); shaded = isolated, dashed = threshold",
             x=0.01, ha="left")
fig.tight_layout()

Sanity check: one random isolated frame per fish, with all masks drawn and a circle of radius `MIN_DIST` around the isolated fish's centroid. A circle drawn from the centroid is only a rough guide; the actual test uses the mask outline.

In [ ]:
def full_mask(fr, i):
    y0, y1, x0, x1 = fr["boxes"][i]
    m = np.zeros(d["shape"], bool)
    m[y0:y1, x0:x1] = np.unpackbits(fr["masks"][i])[: (y1 - y0) * (x1 - x0)].reshape(y1 - y0, x1 - x0)
    return m


rng = np.random.default_rng(0)
fig, axes = plt.subplots(1, len(FISH), figsize=(16, 4.2))
for ax, lab in zip(axes, FISH):
    ax.set_title(NAMES[lab])
    ax.set(xlim=(0, d["shape"][1]), ylim=(d["shape"][0], 0), xticks=[], yticks=[], aspect="equal")
    if len(isolated[lab]) == 0:
        continue
    k = rng.choice(isolated[lab])
    fr = d["frames"][k]
    for i, l in enumerate(fr["labels"]):
        if l == 4:
            continue
        ys, xs = np.nonzero(full_mask(fr, i))
        ax.scatter(xs, ys, s=0.2, color=COLORS[l])
        if l == lab:
            ax.add_patch(plt.Circle((xs.mean(), ys.mean()), MIN_DIST, fill=False, ls="--", color="k"))
    ax.set_xlabel(f"frame {k}")
fig.tight_layout()

## Save

In [ ]:
pd.DataFrame({"frame": np.arange(n_frames), "all_tracked": all_tracked,
              **{f"nearest_px_{l}": nearest[:, i] for i, l in enumerate(FISH)},
              **{f"isolated_{l}": nearest[:, i] >= MIN_DIST for i, l in enumerate(FISH)}}
             ).to_csv(out / f"{TAG}_isolation.csv", index=False)
eps.to_csv(out / f"{TAG}_isolation_episodes.csv")
print("saved to", out)

## Pairs of fish at the same spot while isolated

For each pair of fish (A, B), find an isolated frame of A and an isolated frame of B (any two times in the recording) in which A and B sit in almost the same place:

* **centroid** within `POS_TOL` px, and
* **heading** (tail → head direction, 0–360°) within `ANGLE_TOL` degrees, so a fish lying on the same line but facing the other way does **not** match.

The head is the end of the body axis with more mask pixels on its side: the head and trunk are bulky, while the tail end tapers to a thin caudal peduncle. This is the same rule `fish_midline_from_mask` uses in the compare_electrode notebook. On 5,991 masks from this recording it agrees with the SLEAP head/tail keypoints 98.1% of the time.

Frames inside one isolation episode are nearly identical, so the matches are boiled down to **the closest match for each pair of episodes** (one episode of A, one of B).

In [ ]:
from itertools import combinations

POS_TOL = 15      # px, max centroid distance
ANGLE_TOL = 15    # deg, max heading difference


def pose(fr, lab):
    """Centroid (x, y) and heading (tail -> head, deg, 0-360) of fish `lab` in frame `fr`."""
    i = fr["labels"].index(lab)
    y0, y1, x0, x1 = fr["boxes"][i]
    ys, xs = np.nonzero(np.unpackbits(fr["masks"][i])[: (y1 - y0) * (x1 - x0)].reshape(y1 - y0, x1 - x0))
    xs, ys = xs + x0, ys + y0
    cx, cy = xs.mean(), ys.mean()
    v = np.linalg.eigh(np.cov(xs, ys))[1][:, 1]               # major principal axis
    t = (xs - cx) * v[0] + (ys - cy) * v[1]                   # position along the axis
    if (t > 0).sum() < (t < 0).sum():                         # head = bulkier half
        v = -v
    return cx, cy, np.degrees(np.arctan2(v[1], v[0])) % 360


poses = {}
for lab in FISH:
    p = pd.DataFrame([pose(d["frames"][k], lab) for k in isolated[lab]],
                     columns=["x", "y", "angle"], index=isolated[lab])
    p["episode"] = np.searchsorted(episodes(isolated[lab]).start, p.index, side="right") - 1
    poses[lab] = p


def angle_diff(a, b):
    d = np.abs(a - b) % 360
    return np.minimum(d, 360 - d)


best = []
for a, b in combinations(FISH, 2):
    pa, pb = poses[a], poses[b]
    near = cKDTree(pa[["x", "y"]]).sparse_distance_matrix(
        cKDTree(pb[["x", "y"]]), POS_TOL, output_type="coo_matrix")
    m = pd.DataFrame({"fish_a": a, "fish_b": b,
                      "frame_a": pa.index[near.row], "frame_b": pb.index[near.col],
                      "episode_a": pa.episode.values[near.row], "episode_b": pb.episode.values[near.col],
                      "dist_px": near.data,
                      "angle_diff": angle_diff(pa.angle.values[near.row], pb.angle.values[near.col])})
    m = m[m.angle_diff <= ANGLE_TOL]
    best.append(m.loc[m.groupby(["episode_a", "episode_b"]).dist_px.idxmin()] if len(m) else m)
best = pd.concat(best, ignore_index=True).sort_values(["fish_a", "fish_b", "dist_px"])

summary = (best.groupby(["fish_a", "fish_b"]).size().rename("matched episode pairs")
           .reindex(pd.MultiIndex.from_tuples(combinations(FISH, 2), names=["fish_a", "fish_b"]), fill_value=0))
summary

In [ ]:
best.head(10).round(1)

Where the matches are. Each marker is one matched episode pair: the fill colour is fish A and the outline colour is fish B. Grey shows all isolated positions.

In [ ]:
pairs = list(combinations(FISH, 2))
fig, axes = plt.subplots(2, 3, figsize=(13, 9))
bg = pd.concat(poses.values())
for ax, (a, b) in zip(axes.flat, pairs):
    m = best[(best.fish_a == a) & (best.fish_b == b)]
    ax.scatter(bg.x, bg.y, s=0.3, color="0.85", rasterized=True)
    ax.scatter(poses[a].x[m.frame_a], poses[a].y[m.frame_a], s=40,
               color=COLORS[a], edgecolor=COLORS[b], lw=2)
    ax.set(xlim=(0, d["shape"][1]), ylim=(d["shape"][0], 0), xticks=[], yticks=[], aspect="equal",
           title=f"{NAMES[a]}  vs  {NAMES[b]}  ({len(m)})")
fig.tight_layout()

Sanity check: the closest match for each pair, with both masks drawn on top of each other.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(13, 8.5))
for ax, (a, b) in zip(axes.flat, pairs):
    ax.set(xticks=[], yticks=[], aspect="equal", title=f"{NAMES[a]}  vs  {NAMES[b]}")
    m = best[(best.fish_a == a) & (best.fish_b == b)]
    if m.empty:
        ax.text(0.5, 0.5, "no match", ha="center", transform=ax.transAxes)
        continue
    r = m.iloc[0]
    for lab, k in [(a, int(r.frame_a)), (b, int(r.frame_b))]:
        fr = d["frames"][k]
        ys, xs = np.nonzero(full_mask(fr, fr["labels"].index(lab)))
        ax.scatter(xs, ys, s=1, color=COLORS[lab], alpha=0.5, label=f"{NAMES[lab]} @ frame {k}")
    cx, cy = poses[a].x[r.frame_a], poses[a].y[r.frame_a]
    ax.set(xlim=(cx - 150, cx + 150), ylim=(cy + 150, cy - 150))
    ax.legend(loc="lower left", fontsize=8, markerscale=6)
    ax.set_xlabel(f"{r.dist_px:.1f} px apart, {r.angle_diff:.1f}° heading difference")
fig.tight_layout()

In [ ]:
best.to_csv(out / f"{TAG}_isolated_position_matches.csv", index=False)
print("saved", out / f"{TAG}_isolated_position_matches.csv")

## EOD voltages of matched fish pairs

For each position match above, take one EOD from each fish and overlay the voltages recorded on all 10 electrode pairs.

* **EOD data**: `Data_4fish_<recording>.csv` has one row per detected EOD, with 10 electrode pairs × 25 samples. `Frame - 1` is the mask frame index (the compare_electrode notebook checked this). The sampling rate is taken as 20 kHz, the value that notebook uses.
* **The file doesn't say which fish emitted each EOD.** EODs are attributed with the same rule as `attribute_eod_fish` in the compare_electrode notebook: the electrode pair with the largest amplitude should be nearest the emitter, so the EOD is credited to the fish whose centroid is closest to that pair's midpoint. Isolation makes this more reliable, because the other fish are at least `MIN_DIST` px away.
* For each matched frame, the notebook uses an EOD attributed to that fish within ±1 frame: the one closest in time, with ties going to the largest amplitude.

In [ ]:
FS = 20_000                                                   # Hz, as in the compare_electrode notebook
EOD_CSV = f"data/Data_4fish_{REC.removeprefix('video')}.csv"
ELECTRODES = "data/electrodes_signed_px_coord_2024-02-13.csv"

el = pd.read_csv(ELECTRODES)
PAIRS = el.pair.tolist()
pair_mid = np.c_[(el.x1 + el.x2) / 2, (el.y1 + el.y2) / 2]   # (x, y) video pixels

cols = [f"electrode_{p}_time_{t}" for p in PAIRS for t in range(1, 26)]
eod = pd.read_csv(EOD_CSV, usecols=["Frame"] + cols)
V = eod[cols].to_numpy(float).reshape(len(eod), len(PAIRS), 25)
V -= V.mean(axis=2, keepdims=True)                            # demean each trace
eod_frame = eod.Frame.to_numpy() - 1                          # mask frame index
amp = np.abs(V).max(axis=2)                                   # (n_eod, n_pairs)
rows_by_frame = pd.DataFrame({"f": eod_frame}).groupby("f").indices
print(f"{len(eod)} EODs, {len(PAIRS)} electrode pairs")

In [ ]:
def emitter(i):
    """Fish credited with EOD i: centroid closest to the midpoint of its strongest electrode pair."""
    fr = d["frames"][eod_frame[i]]
    labs = [l for l in fr["labels"] if l in FISH]
    xy = np.array([pose(fr, l)[:2] for l in labs])
    return labs[np.argmin(np.hypot(*(xy - pair_mid[amp[i].argmax()]).T))]


def eod_of(lab, k):
    """An EOD attributed to fish `lab` within ±1 frame of frame k (closest in time, then largest), or None."""
    cands = [i for dk in (0, -1, 1) for i in rows_by_frame.get(k + dk, []) if emitter(i) == lab]
    return max(cands, key=lambda i: (-abs(eod_frame[i] - k), amp[i].max()), default=None)


best["eod_a"] = [eod_of(r.fish_a, r.frame_a) for r in best.itertuples()]
best["eod_b"] = [eod_of(r.fish_b, r.frame_b) for r in best.itertuples()]
eod_pairs = best.dropna(subset=["eod_a", "eod_b"]).astype({"eod_a": int, "eod_b": int})
eod_pairs["corr"] = [np.corrcoef(V[a].ravel(), V[b].ravel())[0, 1]
                     for a, b in zip(eod_pairs.eod_a, eod_pairs.eod_b)]

(eod_pairs.groupby(["fish_a", "fish_b"])
 .agg(matches_with_both_eods=("corr", "size"), median_waveform_corr=("corr", "median"))
 .reindex(summary.index).fillna({"matches_with_both_eods": 0}).round(2))

`median_waveform_corr` is the correlation between the two fish's full 10-pair × 25-sample voltage patterns. A high value means the two EODs look alike on the electrodes when the fish are at the same spot.

In [ ]:
def plot_match(r):
    """Tank map of both fish + electrode pairs, and both EODs overlaid on each of the 10 pairs."""
    fa, fb, ka, kb, ia, ib = (int(r[c]) for c in ["fish_a", "fish_b", "frame_a", "frame_b", "eod_a", "eod_b"])
    fig = plt.figure(figsize=(16, 5.5))
    gs = fig.add_gridspec(2, 7)
    ax0 = fig.add_subplot(gs[:, :2])
    for lab, k in [(fa, ka), (fb, kb)]:
        fr = d["frames"][k]
        ys, xs = np.nonzero(full_mask(fr, fr["labels"].index(lab)))
        ax0.scatter(xs, ys, s=0.3, color=COLORS[lab], label=f"{NAMES[lab]}, frame {k}")
    for e in el.itertuples():
        ax0.plot([e.x1, e.x2], [e.y1, e.y2], color="0.6", lw=0.8)
        ax0.text(e.x1 + 0.3 * (e.x2 - e.x1), e.y1 + 0.3 * (e.y2 - e.y1), e.pair, fontsize=8,
                 ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.5))
    ax0.set(xlim=(0, d["shape"][1]), ylim=(d["shape"][0], 0), aspect="equal", xticks=[], yticks=[])
    ax0.legend(loc="upper center", bbox_to_anchor=(0.5, 0), fontsize=8, markerscale=10, frameon=False)

    t_ms = np.arange(25) / FS * 1e3
    lim = 1.05 * np.abs(V[[ia, ib]]).max()
    for j, pid in enumerate(PAIRS):
        ax = fig.add_subplot(gs[j // 5, 2 + j % 5])
        for lab, i in [(fa, ia), (fb, ib)]:
            ax.plot(t_ms, V[i, j], color=COLORS[lab], lw=1.5)
        ax.axhline(0, color="0.85", lw=0.5)
        ax.set(title=f"pair {pid}", ylim=(-lim, lim))
        ax.spines[["top", "right"]].set_visible(False)
        if j % 5:
            ax.set_yticklabels([])
        if j < 5:
            ax.set_xticklabels([])
        if j == 5:
            ax.set(xlabel="time (ms)", ylabel="voltage (recorded units, demeaned)")
    fig.suptitle(f"{NAMES[fa]} vs {NAMES[fb]}: centroids {r.dist_px:.1f} px apart, "
                 f"headings {r.angle_diff:.1f}° apart, waveform correlation {r['corr']:.2f}",
                 x=0.01, ha="left")
    fig.tight_layout()


plot_match(eod_pairs.sort_values("dist_px").iloc[0])

Any other match can be plotted with `plot_match(eod_pairs.iloc[i])`. Save the table of matched EOD pairs:

In [ ]:
eod_pairs.to_csv(out / f"{TAG}_isolated_match_eods.csv", index=False)
print("saved", out / f"{TAG}_isolated_match_eods.csv")

## Sanity check: is each EOD attributed to the right fish?

The electrode rule above only looks at which electrode pair is loudest. To check it, every EOD used in a match is compared with the field model from `eod_two_layer_pipeline_compare_electrode.ipynb`:

1. For **each fish** in the EOD's frame, `solve_instant_field` computes the potential with that fish emitting a unit current while the other fish act as two-layer (skin + interior) bodies. Sampling the field at the electrodes gives that fish's predicted voltage pattern **v** on the 10 pairs.
2. A resistive field has a fixed spatial pattern scaled by the EOD current, so the recorded EOD **V** (10 pairs × 25 samples) should be close to **v**·I(t) for the true emitter. The only rescaling is one free amplitude per time sample. **R²** is the fraction of the recorded signal that this explains. It doesn't depend on sign, so the choice of EOD phase doesn't matter.
3. An attribution counts as **easy** when the electrode rule and the best-fitting fish agree, the best R² ≥ `R2_MIN`, and the runner-up fish's R² is at least `R2_MARGIN` lower (no other fish could plausibly have produced the pattern).

In [ ]:
import json
import re
import scipy.sparse as sp
from scipy.sparse.linalg import splu
from scipy.ndimage import zoom
from scipy.interpolate import UnivariateSpline

PIPELINE = "eod_two_layer_pipeline_compare_electrode.ipynb"
NEEDED = {"fish_midline_from_mask", "gammas", "build_A_shell", "build_multipole_b",
          "load_electrode_pairs", "solve_instant_field"}
for c in json.load(open(PIPELINE))["cells"]:            # run only the cells that define the model
    src = "".join(c["source"])
    if c["cell_type"] == "code" and NEEDED & set(re.findall(r"^def (\w+)", src, re.M)):
        exec(src)

N_GRID, DELTA_H, N_POLES = 200, 0.5, 100                 # same settings as the pipeline
R2_MIN, R2_MARGIN = 0.8, 0.3

model_pairs = load_electrode_pairs(ELECTRODES)          # (pair, + electrode, - electrode) in model coords
assert [p for p, _, _ in model_pairs] == PAIRS


def sample(phi, p):
    """Bilinear sample of an (N, N) model field at tank point p."""
    fi, fj = np.asarray(p) * (N_GRID - 1)
    i, j = min(int(fi), N_GRID - 2), min(int(fj), N_GRID - 2)
    a, b = fi - i, fj - j
    return (1-a)*(1-b)*phi[i, j] + a*(1-b)*phi[i+1, j] + (1-a)*b*phi[i, j+1] + a*b*phi[i+1, j+1]


def predicted_patterns(k):
    """{fish: unit-current voltage pattern on the electrode pairs} for every fish in frame k."""
    fr = d["frames"][k]
    labs = [l for l in fr["labels"] if l in FISH]
    masks = np.stack([full_mask(fr, fr["labels"].index(l)) for l in labs])
    out = {}
    for s, lab in enumerate(labs):
        phi = solve_instant_field(masks, N_GRID, s, 1.0, DELTA_H, N_POLES, **MAT)[0]
        out[lab] = np.array([sample(phi, pp) - sample(phi, pm) for _, pp, pm in model_pairs])
    return out


def fit_r2(Vrec, v):
    """Fraction of a recorded EOD (pairs x samples) explained by pattern v times a free waveform I(t)."""
    u = v / np.linalg.norm(v)
    return ((u @ Vrec) ** 2).sum() / (Vrec ** 2).sum()

Solve the model for every EOD used in a match. This takes about 3 s per EOD (one solve per fish), so results are cached and only missing EODs are computed on later runs.

In [ ]:
check_eods = np.unique(np.r_[eod_pairs.eod_a, eod_pairs.eod_b])
cache = out / f"{REC}_predicted_patterns.pkl"
patterns = pickle.load(open(cache, "rb")) if cache.exists() else {}
for i in check_eods:
    if i not in patterns:
        patterns[i] = predicted_patterns(eod_frame[i])
pickle.dump(patterns, open(cache, "wb"))

rows = []
for i in check_eods:
    r2 = {lab: fit_r2(V[i], v) for lab, v in patterns[i].items()}
    ranked = sorted(r2, key=r2.get, reverse=True)
    rows.append(dict(eod=i, rule_fish=emitter(i), model_fish=ranked[0],
                     r2_best=r2[ranked[0]], r2_second=r2[ranked[1]]))
check = pd.DataFrame(rows).set_index("eod")
check["agree"] = check.rule_fish == check.model_fish
check["easy"] = check.agree & (check.r2_best >= R2_MIN) & (check.r2_best - check.r2_second >= R2_MARGIN)
print(f"{len(check)} EODs checked: rule and model agree for {check.agree.mean():.0%}, "
      f"easy attribution for {check.easy.mean():.0%}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4.5))
for (agree, easy), g in check.groupby(["agree", "easy"]):
    label = "easy" if easy else ("rule = model, not clear-cut" if agree else "rule ≠ model")
    ax.scatter(g.r2_second, g.r2_best, s=12, label=f"{label} ({len(g)})",
               color="#2a78d6" if easy else ("0.6" if agree else "#d1495b"))
ax.axhline(R2_MIN, color="k", ls="--", lw=0.8)
ax.plot([0, 1 - R2_MARGIN], [R2_MARGIN, 1], color="k", ls=":", lw=0.8)
ax.set(xlabel="R² of the runner-up fish", ylabel="R² of the best-fitting fish", xlim=(0, 1), ylim=(0, 1))
ax.legend(fontsize=8, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

A closer look at single EODs. Left: all fish in the EOD's frame. Right: the recorded voltage on each pair at the EOD's peak sample (grey bars) and each fish's predicted pattern, scaled to fit best (lines).

In [ ]:
def plot_attribution(i, ax_map, ax_bar):
    k = eod_frame[i]
    fr = d["frames"][k]
    t = np.abs(V[i]).max(0).argmax()
    for lab in patterns[i]:
        ys, xs = np.nonzero(full_mask(fr, fr["labels"].index(lab)))
        ax_map.scatter(xs, ys, s=0.2, color=COLORS[lab])
    for e in el.itertuples():
        ax_map.plot([e.x1, e.x2], [e.y1, e.y2], color="0.7", lw=0.6)
        ax_map.text(e.x1 + 0.3 * (e.x2 - e.x1), e.y1 + 0.3 * (e.y2 - e.y1), e.pair, fontsize=7,
                    ha="center", va="center", bbox=dict(fc="white", ec="none", pad=0.3))
    ax_map.set(xlim=(0, d["shape"][1]), ylim=(d["shape"][0], 0), aspect="equal", xticks=[], yticks=[])
    x = np.arange(len(PAIRS))
    ax_bar.bar(x, V[i][:, t], color="0.8", label="recorded")
    for lab, v in patterns[i].items():
        scale = v @ V[i][:, t] / (v @ v)
        ax_bar.plot(x, scale * v, "o-", color=COLORS[lab], ms=4, lw=1.2,
                    label=f"{NAMES[lab]}: R² = {fit_r2(V[i], v):.2f}")
    ax_bar.axhline(0, color="0.5", lw=0.5)
    ax_bar.set_xticks(x, PAIRS)
    ax_bar.set_xlabel("electrode pair")
    ax_bar.legend(fontsize=7, loc="best")
    ax_bar.spines[["top", "right"]].set_visible(False)
    c = check.loc[i]
    ax_bar.set_title(f"EOD {i}, frame {k}: rule → {NAMES[c.rule_fish]}, model → {NAMES[c.model_fish]}"
                     f"{'  (easy)' if c.easy else ''}", fontsize=9)


def plot_attributions(eods):
    fig, axes = plt.subplots(len(eods), 2, figsize=(12, 3.6 * len(eods)),
                             gridspec_kw=dict(width_ratios=[1, 2.6]), squeeze=False)
    for i, (am, ab) in zip(eods, axes):
        plot_attribution(i, am, ab)
    fig.tight_layout()


ex = eod_pairs.sort_values("dist_px").iloc[0]                # the example match plotted above
plot_attributions([int(ex.eod_a), int(ex.eod_b)])

A random sample of other checked EODs: three easy and three that aren't.

In [ ]:
rng = np.random.default_rng(1)
easy_ids, hard_ids = check.index[check.easy].to_numpy(), check.index[~check.easy].to_numpy()
plot_attributions(list(rng.choice(easy_ids, min(3, len(easy_ids)), replace=False))
                  + list(rng.choice(hard_ids, min(3, len(hard_ids)), replace=False)))

## Peak amplitude ratio of matched fish pairs

For every matched pair of frames (fish A and fish B at the same spot with the same heading, both isolated) where **both** EODs are easy attributions, the ratio is the peak absolute recorded voltage of A's EOD divided by B's. The peak is taken over all 10 pairs and 25 samples, with A the lower label. Both fish sit at the same place relative to the electrodes, so this ratio compares the two fish's EOD strength directly.

In [ ]:
peak = np.abs(V).max(axis=(1, 2))                        # peak |voltage| per EOD
clean = eod_pairs[eod_pairs.eod_a.map(check.easy) & eod_pairs.eod_b.map(check.easy)].copy()
clean["amp_ratio"] = peak[clean.eod_a] / peak[clean.eod_b]

amp_ratio = clean.amp_ratio.to_numpy()                                       # all pairs together
amp_ratio_by_pair = {ab: g.amp_ratio.to_numpy() for ab, g in clean.groupby(["fish_a", "fish_b"])}

print(f"{len(amp_ratio)} matched frame pairs with both EODs cleanly attributed")
clean.groupby(["fish_a", "fish_b"]).amp_ratio.describe()[["count", "25%", "50%", "75%"]].round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for n, ((a, b), r) in enumerate(amp_ratio_by_pair.items()):
    ax.scatter(np.log2(r), np.full(len(r), n) + rng.uniform(-0.15, 0.15, len(r)), s=10,
               color=COLORS[a], edgecolor=COLORS[b], lw=1)
    ax.plot(np.log2(np.median(r)), n, "k|", ms=18, mew=2)
ax.set_yticks(range(len(amp_ratio_by_pair)), [f"{NAMES[a]} / {NAMES[b]}" for a, b in amp_ratio_by_pair])
ax.axvline(0, color="0.6", lw=0.8)
ticks = np.arange(-3, 4)
ax.set_xticks(ticks, [f"{2.0 ** t:g}" for t in ticks])
ax.set_xlabel("peak amplitude ratio A / B (log scale); black bar = median")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()

In [ ]:
clean.to_csv(out / f"{TAG}_clean_match_amp_ratios.csv", index=False)
np.save(out / f"{TAG}_amp_ratio.npy", amp_ratio)
print("saved", out / f"{TAG}_clean_match_amp_ratios.csv", "and", out / f"{TAG}_amp_ratio.npy")